<a href="https://colab.research.google.com/github/farahdinaariff/Data-Science-Project/blob/main/README.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TropicalPV: Probabilistic Solar Power Forecasting Under Tropical Climate Uncertainty

**Notebook:** `TropicalPV_Probabilistic_Solar_Forecasting_v11_VIVA.ipynb`
**Author:** Farah Dina Bte Esa Mahmud Ariff
**Programme:** MSc Data Science, University of Hertfordshire (PSB Academy), module 7FTC2001

This notebook compares four deep-learning forecasters of hourly solar irradiance at Darwin, northern Australia (tropical savanna, Köppen Aw). Two of them give a single value (LSTM, GRU) and two give a value plus a prediction range (MC-Dropout LSTM, Quantile Regression GRU). Each model is scored on two separate questions: how accurate the single value is, and whether the stated 80% range actually contains the outcome about 80% of the time, in the wet season as well as the dry.

## Research question

> Which deep learning architecture, deterministic (LSTM, GRU) or probabilistic (MC-Dropout LSTM, Quantile Regression GRU), produces the most accurate and best-calibrated solar irradiance forecasts for tropical climates under wet/dry seasonal conditioning?

| Objective | Focus |
|---|---|
| **O1** Accuracy | Skill against smart persistence; matched irradiance-vs-PV proxy test at Alice Springs |
| **O2** Calibration | CRPS, pinball loss, PICP, MIW; whether uncertainty quantification costs accuracy in the wet season |
| **O3** Season and ramps | Effect of wet/dry conditioning; behaviour during sudden cloud ramps |
| **O4** Decision support | Whether interval width can serve as an operating threshold (PICP at nominal, MIW ≤ X) |
| **O5** Transfer | Zero-shot cross-climate transfer to other tropical and equatorial sites |

## Headline results (FULL run, 4 test periods × 5 seeds)

| Result | Finding |
|---|---|
| Accuracy | All four networks beat persistence by roughly 8 to 12%. QR-GRU, GRU and LSTM are statistically tied. |
| Reliability | Only QR-GRU's 80% range held near nominal in both seasons. Wet-season coverage was 0.795 for QR-GRU and 0.446 for MC-Dropout LSTM. |
| Limits | Coverage fell to about 46% during sudden cloud ramps (around 6% of hours) for every model. No width-based operating rule was stable. |
| Transfer | Applied to other sites without recalibration, coverage ranged from 0.26 to 0.89. |
| Data integrity | The DKASC "local" timestamp is UTC+9:00, not Darwin's UTC+9:30. Left uncorrected, the 30-minute error made QR-GRU look 16.6% better than persistence when it was actually 36% worse in that early controlled test. |

In short, how the uncertainty is modelled matters far more than the choice between LSTM and GRU cells.

## Notebook layout

The notebook has two sections.

**1. VIVA section (top of the notebook).** A 10-minute presentation layer for the viva panel: the hook, problem, insight, solution and conclusion, with Figures H1, V0 to V5 and Tables V0 to V4 plus backup tables. These cells train nothing. They only define names that begin with `VIVA` or `_viva`, and they leave `CFG`, the seeds and all research variables untouched.

Each VIVA figure or table carries one of three labels:

| Label | Meaning |
|---|---|
| **STATIC** | Text or values copied from the executed notebook or the report, with the stage named |
| **SNAPSHOT** | A verified copy of a result table from the original FULL run, shown when the research cells have not been run in this session |
| **LIVE** | The same table read from this session's variables (`R_LEADER`, `R_DM`, `SUBGROUP_R`, `ABLATION`, `HORIZON_R`, `RANK_STAB`, `INV_CMP3`, `INV_SUMMARY`) after a full run |

**2. Research section (starts at "START OF RESEARCH").** The original pipeline, unchanged, organised under navigation headers PART A to PART M. Open the **Table of contents** panel on the left of Colab to jump between them.

| Part | Content | Stages |
|---|---|---|
| A | Project overview, configuration (`CFG`), utilities | 0 to 1 |
| B | Raw file inventory and file-level KEEP / HOLD / DROP audit | 2 to 3 |
| C | Timestamp investigation: two clocks, solar-noon test, effect on models | 4 to 5 |
| D | Production ingestion, validation gate, QC, clear-sky index k_t, features | 6 to 7 |
| E | Exploratory data analysis: sky regimes, ramps, domain shift | 8 |
| F | Feature screen, chronological splits, executed leakage audit | 9 |
| G | Architectures, metrics, naive references | 10 |
| H | Lookback study and equal-budget tuning | 11 |
| I | Last-origin training, conformal calibration, error structure | 12 |
| J | Season and ramp subgroups, PV proxy, cross-climate transfer, decision support, reserve sizing | 13 |
| K | Rolling-origin evaluation (R1 to R6), significance tests, seasonal ablation | 14 |
| L | Findings, research-question synthesis, limitations, exports | 15 to 16 |
| M | Appendix: optional data-download utilities (off by default) | – |

Every stage follows the same chain: analysis, finding, risk, decision rule, modification, re-analysis, before-vs-after. Where a method changed during the project, both versions are generated from the same raw data so the effect can be seen.

## Models compared

| Model | Output | Uncertainty type | Loss |
|---|---|---|---|
| LSTM | Point | None | MSE |
| GRU | Point | None | MSE |
| MC-Dropout LSTM | Point + range (50 stochastic passes) | Epistemic | MSE |
| QR-GRU | 7 quantiles, 5th to 95th percentile | Aleatoric | Pinball |
| HMC-LSTM *(diagnostic only)* | Point + range | Epistemic + aleatoric | Gaussian NLL |
| Persistence, climatology, seasonal naive | Benchmarks | – | – |

**Target:** clear-sky index k_t one hour ahead (measured GHI divided by modelled clear-sky GHI).
**Inputs:** 24 hours of k_t history, solar geometry, hour and day-of-year encodings, and a wet-season flag (16 features).
**Evaluation:** 60 / 12.5 / 12.5 / 15 chronological train / validation / calibration / test split, 4 expanding-window test periods, 5 seeds, the same 12-configuration tuning grid for every model, moving-block bootstrap CIs, and Diebold-Mariano tests with Newey-West variance and Holm correction.

## Data

All data are read from Google Drive. The notebook expects a root folder that contains one or more folders whose names start with `Data_` (or the older `Data-`).

| Role | Source | Folder used in the original run |
|---|---|---|
| Primary target | Darwin NT Class A station (DKASC), 1-second files 2021 to 2023, extended with the same station's 5-minute export | `Data-DKASC Darwin`, `Data_Australia Tropical Class A irradiance-Darwin-Katherine` |
| PV proxy site | Alice Springs DKASC irradiance, meteorology and measured PV | `Data_Australia DKASC Alice Springs` |
| Zero-shot transfer sites | Tennant Creek, Yulara, Singapore (satellite), Gan (Maldives), Petrolina (BSRN), Hong Kong, South Florida (NSRDB) | `Data_Australia Tropical Class A irradiance-Tennant Creek`, `Data_Australia Yulara Solar System`, `Data_NSRDB Himawari PSM v3`, `Data_Singapore Open-Meteo Satellite Radiation`, `Data-Nasa Power Singapore`, `Data_Petrolina BSRN via PANGAEA`, `Data_HKUST Hong Kong`, `Data_World Bank-ESMAP`, `Data-NSRDB US Soutwest` |

The folder names do not have to match exactly. Stage 3 assigns each file a role with regular expressions, and anything it cannot classify is listed as UNCLASSIFIED rather than used silently. Large files are streamed in chunks and never loaded whole.

The data files are not included with the notebook. DKASC data are published openly by the DKA Solar Centre. The appendix (Part M) has optional download cells for Open-Meteo, OpenElectricity and NSRDB, all switched off by default.

## How to run in Google Colab

### Option 1: Present from the saved notebook

All figures and tables in the VIVA section are already saved in the notebook. Open it in Colab and scroll. Nothing needs to run.

### Option 2: Re-run the VIVA section only (seconds)

1. Run the **VIVA Setup** cell (cell 2). It trains nothing and finishes in about a second.
2. Run the VIVA cells from top to bottom. They show the SNAPSHOT tables when the research cells have not been run.

### Option 3: Full reproduction

1. **Runtime → Change runtime type → T4 GPU.** FULL mode needs a GPU.
2. Put the `Data_*` folders in your Google Drive (by default directly under `My Drive`).
3. Run the **Setup — Configuration** cell in Part A and accept the Drive authorisation prompt.
4. Check the printed `Data root` line. If the folders are elsewhere, change `CFG.DRIVE_ROOT`. The notebook also tries to find them on its own.
5. **Runtime → Run all**, or run the research section in order from Part A to Part L.
6. Afterwards, return to the top and run the VIVA cells again. Their tables switch from SNAPSHOT to LIVE.

### Run modes

Set `CFG.MODE` in the configuration cell, or set the `TROPICALPV_MODE` environment variable before running.

| Mode | What changes | Approximate time on a T4 |
|---|---|---|
| `SMOKE` | 3 epochs, 2 seeds, 2 folds, 300,000 rows per file. For checking that the pipeline runs end to end. | Short; a quick end-to-end check |
| `FAST` | Same protocol with 3 seeds, 3 folds and a smaller tuning grid | About 40% of FULL |
| `FULL` | 4 folds × 5 seeds, full tuning grid. Produces the reported results. | About 2.5 hours (8,799 s in the recorded run) |

Expensive stages are checkpointed against a fingerprint of the input files, so a rerun on the same data skips work already done. Set `CFG.FORCE_REBUILD = True` to ignore the checkpoints.

### Safeguards you may run into

- **Timing gate.** If the primary site's clear-day timing lag is more than 10 minutes, the run stops. This is deliberate. Only set `CFG.TIMING_GATE_ENFORCE = False` to inspect a failing run; results from such a run are not valid.
- **Primary site.** If Darwin fails to load, the run stops. It is never replaced by another site.
- **Drive not mounted.** The configuration cell prints a warning and writes nothing to disk. Mount Drive and run the cell again.

## Requirements

Colab's default image has everything needed. No `pip install` cell is required for the main analysis. The recorded FULL run used:

| Package | Version |
|---|---|
| Python | 3.13.15 |
| TensorFlow | 2.20.0 |
| NumPy | 2.1.3 |
| pandas | 2.2.3 |
| SciPy | 1.16.3 |
| matplotlib | 3.10.0 |
| PyArrow | 23.0.1 |
| scikit-learn | Colab default |

The optional OpenElectricity download cell in Part M also needs the `openelectricity` package and an API key stored in Colab Secrets.

## Outputs

Everything is written to `_tropicalpv_artifacts/` inside the data root on Drive.

| File | Content |
|---|---|
| `figures/*.png` | All research figures (33 in the recorded run) |
| `TropicalPV_v7_results.xlsx` | One sheet per result table: leaderboard, Diebold-Mariano tests, rank stability, ablation, horizon sweep, subgroups, calibration, transfer, objective matrix, limitations. Falls back to CSV files if Excel writing fails. |
| `results_registry.json` | Every headline number with its provenance (105 entries). This is the single source of numbers for the written report. |
| Checkpoint files | Cached intermediate results keyed by the data fingerprint |

## Limitations

- One training site (Darwin). The findings are Darwin-specific until repeated at another tropical ground station.
- The target is irradiance, not PV power. The proxy test used an arid site, Alice Springs.
- Hourly resolution averages out ramps inside the hour, so the ramp problem is likely worse for a real plant.
- Inputs are past sunlight only. No model can see cloud coming, which is why all of them fail during ramps.
- Test periods rely on the five-minute export. It agrees closely with the one-second data (R² = 0.9987 over 4,718 hours), but a sensor change after 2023 cannot be ruled out.
- Seasonal comparisons were not corrected for multiple testing.
- The planned Singapore monsoon training test was not carried out.

## Citation

Ariff, F. D. B. E. M. (2026). *Probabilistic Solar Power Forecasting Under Tropical Climate Uncertainty: A Comparative Study of Deterministic vs Uncertainty-Quantifying Deep Learning Architectures on Tropical and Equatorial Datasets.* MSc Data Science dissertation (7FTC2001), University of Hertfordshire / PSB Academy.